# Rank LSTM Definition and Training
This section introduces the Rank-LSTM (RLSTM), a significant enhancement over the baseline model.

* RankLSTMModel: The model architecture itself is identical to the baseline LSTM. The key innovation is not in the architecture but in the training objective.

* RankLSTM: This wrapper class implements the novel training procedure. The compute_losses method defines a composite loss function that includes two parts:

    1. Regression Loss (MSE): The same as the baseline, encouraging the model to predict the correct return value.

    2. Pairwise Ranking Loss: This is the crucial addition. It compares pairs of stocks and penalizes the model if their predicted rank order (which stock will perform better) does not match the actual rank order.

This is the loss formula used:
$$
\begin{equation*}
\mathcal{L}(\hat{\mathbf{f}}^{(t+1)}, \mathbf{r}^{(t+1)}) = 
\underbrace{
    \|\hat{\mathbf{f}}^{(t+1)} - \mathbf{r}^{(t+1)}\|_2^2
}_{\text{Pointwise Regression Loss}} 
+ \alpha 
\underbrace{
    \sum_{i} \sum_{j} \max\left(0, -(\hat{f}_i^{(t+1)} - \hat{f}_j^{(t+1)}) (r_i^{(t+1)} - r_j^{(t+1)})\right)
}_{\text{Pairwise Ranking Loss}}
\end{equation*}
$$

In [1]:
import argparse
import copy
import numpy as np
import os
import sys
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from time import time
import math
import scipy.stats as sps
from sklearn.metrics import mean_squared_error, mean_absolute_error

sys.path.append(os.path.abspath('../../'))
from models.evaluate import evaluate
from models.data_loading import load_EOD_data, load_relation_data

In [2]:
seed = 123456789
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
class RankLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super(RankLSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_units, batch_first=True)
        self.fc = nn.Linear(hidden_units, 1)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # Shape: [batch, seq_len, hidden]
        last_output = lstm_out[:, -1, :]  # Last time step
        raw_pred = self.fc(last_output)
        pred = self.leaky_relu(raw_pred)
        return pred


In [4]:
class RankLSTM:
    def __init__(self, data_path, market_name, tickers_fname, parameters,
                 steps=1, epochs=50, batch_size=None, gpu=False):
        self.data_path = data_path
        self.market_name = market_name
        self.tickers_fname = tickers_fname
        self.tickers = np.genfromtxt(os.path.join(data_path, '..', tickers_fname),
                                     dtype=str, delimiter='\t', skip_header=False)
        print('#tickers selected:', len(self.tickers))
        
        # Load the RAW prices for loss calculation
        raw_price_path = os.path.join(data_path, f'{market_name}_raw_prices.npy')
        self.raw_price_data = np.load(raw_price_path)
        if self.market_name == 'NASDAQ':
            self.raw_price_data = self.raw_price_data[:, :-1]
        print('Raw prices shape:', self.raw_price_data.shape)

        self.eod_data, self.mask_data, self.gt_data, _ = \
            load_EOD_data(data_path, market_name, self.tickers, self.raw_price_data, steps)
        
        self.parameters = copy.copy(parameters)
        self.steps = steps
        self.epochs = epochs
        self.batch_size = len(self.tickers) if batch_size is None else batch_size
        self.valid_index = 756
        self.test_index = 1008
        self.trade_dates = self.mask_data.shape[1]
        self.fea_dim = 5
        self.gpu = gpu

        # Set device
        self.device = torch.device('cuda' if gpu and torch.cuda.is_available() else 'cpu')
        print('device:', self.device)

    def get_batch(self, offset=None):
        if offset is None:
            offset = random.randrange(0, self.valid_index)
        seq_len = self.parameters['seq']
        mask_batch = self.mask_data[:, offset: offset + seq_len + self.steps]
        mask_batch = np.min(mask_batch, axis=1)

        base_price_batch = self.raw_price_data[:, offset + seq_len - 1]
        
        # If a raw price is -1234 (or 0), its corresponding mask should be 0
        for i in range(len(mask_batch)):
            if base_price_batch[i] < 1e-8: # If price is 0 or placeholder
                mask_batch[i] = 0.0
        
        return self.eod_data[:, offset:offset + seq_len, :], \
               np.expand_dims(mask_batch, axis=1), \
               np.expand_dims(base_price_batch, axis=1), \
               np.expand_dims(self.gt_data[:, offset + seq_len + self.steps - 1], axis=1)

    def compute_losses(self, pred, base_price, ground_truth, mask, alpha):
        return_ratio = (pred - base_price) / base_price
    
        # Regression loss: MSE with mask
        reg_loss = F.mse_loss(return_ratio * mask, ground_truth * mask)
    
        # Pairwise ranking loss
        pre_pw_dif = return_ratio - return_ratio.t()
        gt_pw_dif = ground_truth - ground_truth.t()
        mask_pw = mask @ mask.t()
        rank_loss = torch.mean(F.relu(-(pre_pw_dif * gt_pw_dif) * mask_pw))
    
        total_loss = reg_loss + alpha * rank_loss
        return total_loss, reg_loss, rank_loss, return_ratio

    def train(self):
        model = RankLSTMModel(self.fea_dim, self.parameters['unit']).to(device)
        optimizer = optim.Adam(model.parameters(), lr=self.parameters['lr'])

        best_valid_pred = np.zeros([len(self.tickers), self.test_index - self.valid_index], dtype=float)
        best_valid_gt = np.zeros_like(best_valid_pred)
        best_valid_mask = np.zeros_like(best_valid_pred)

        best_test_pred = np.zeros([len(self.tickers), self.trade_dates - self.parameters['seq'] -
                                   self.test_index - self.steps + 1], dtype=float)
        best_test_gt = np.zeros_like(best_test_pred)
        best_test_mask = np.zeros_like(best_test_pred)

        best_valid_perf = {'mse': np.inf}
        best_test_perf = {'mse': np.inf}
        best_valid_loss = np.inf

        for epoch in range(self.epochs):
            t1 = time()
            model.train()
            total_loss, total_reg_loss, total_rank_loss = 0.0, 0.0, 0.0

            batch_offsets = np.arange(0, self.valid_index)
            np.random.shuffle(batch_offsets)

            for j in range(self.valid_index - self.parameters['seq'] - self.steps + 1):
                eod_batch, mask_batch, price_batch, gt_batch = self.get_batch(batch_offsets[j])
                x = torch.tensor(eod_batch, dtype=torch.float32, device=device)
                mask = torch.tensor(mask_batch, dtype=torch.float32, device=device)
                base_price = torch.tensor(price_batch, dtype=torch.float32, device=device)
                gt = torch.tensor(gt_batch, dtype=torch.float32, device=device)

                optimizer.zero_grad()
                pred = model(x)
                loss, reg_loss, rank_loss, _ = self.compute_losses(pred, base_price, gt, mask, self.parameters['alpha'])
                loss.backward()
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                total_loss += loss.item()
                total_reg_loss += reg_loss.item()
                total_rank_loss += rank_loss.item()

            denom = self.valid_index - self.parameters['seq'] - self.steps + 1
            print('Train Loss:',
                  total_loss / denom,
                  total_reg_loss / denom,
                  total_rank_loss / denom)

            # Validation
            model.eval()
            with torch.no_grad():
                val_pred = np.zeros([len(self.tickers), self.test_index - self.valid_index], dtype=float)
                val_gt = np.zeros_like(val_pred)
                val_mask = np.zeros_like(val_pred)
                val_loss = val_reg_loss = val_rank_loss = 0.0

                for offset in range(self.valid_index - self.parameters['seq'] - self.steps + 1,
                                    self.test_index - self.parameters['seq'] - self.steps + 1):
                    eod_batch, mask_batch, price_batch, gt_batch = self.get_batch(offset)
                    x = torch.tensor(eod_batch, dtype=torch.float32, device=device)
                    mask = torch.tensor(mask_batch, dtype=torch.float32, device=device)
                    base_price = torch.tensor(price_batch, dtype=torch.float32, device=device)
                    gt = torch.tensor(gt_batch, dtype=torch.float32, device=device)

                    pred = model(x)
                    loss, reg_loss, rank_loss, return_ratio = self.compute_losses(pred, base_price, gt, mask,
                                                                             self.parameters['alpha'])

                    val_loss += loss.item()
                    val_reg_loss += reg_loss.item()
                    val_rank_loss += rank_loss.item()
                    idx = offset - (self.valid_index - self.parameters['seq'] - self.steps + 1)
                    val_pred[:, idx] = return_ratio.squeeze().cpu().numpy()
                    val_gt[:, idx] = gt.squeeze().cpu().numpy()
                    val_mask[:, idx] = mask.squeeze().cpu().numpy()

                denom = self.test_index - self.valid_index
                print('Valid MSE:',
                      val_loss / denom,
                      val_reg_loss / denom,
                      val_rank_loss / denom)
                cur_valid_perf = evaluate(val_pred, val_gt, val_mask)
                print('\tValid performance:', cur_valid_perf)

                # Testing
                test_pred = np.zeros([len(self.tickers), self.trade_dates - self.test_index], dtype=float)
                test_gt = np.zeros_like(test_pred)
                test_mask = np.zeros_like(test_pred)
                test_loss = test_reg_loss = test_rank_loss = 0.0

                for offset in range(self.test_index - self.parameters['seq'] - self.steps + 1,
                                    self.trade_dates - self.parameters['seq'] - self.steps + 1):
                    eod_batch, mask_batch, price_batch, gt_batch = self.get_batch(offset)
                    x = torch.tensor(eod_batch, dtype=torch.float32, device=device)
                    mask = torch.tensor(mask_batch, dtype=torch.float32, device=device)
                    base_price = torch.tensor(price_batch, dtype=torch.float32, device=device)
                    gt = torch.tensor(gt_batch, dtype=torch.float32, device=device)

                    pred = model(x)
                    loss, reg_loss, rank_loss, return_ratio = self.compute_losses(pred, base_price, gt, mask,
                                                                             self.parameters['alpha'])

                    test_loss += loss.item()
                    test_reg_loss += reg_loss.item()
                    test_rank_loss += rank_loss.item()
                    idx = offset - (self.test_index - self.parameters['seq'] - self.steps + 1)
                    test_pred[:, idx] = return_ratio.squeeze().cpu().numpy()
                    test_gt[:, idx] = gt.squeeze().cpu().numpy()
                    test_mask[:, idx] = mask.squeeze().cpu().numpy()

                denom = self.trade_dates - self.test_index
                print('Test MSE:',
                      test_loss / denom,
                      test_reg_loss / denom,
                      test_rank_loss / denom)
                cur_test_perf = evaluate(test_pred, test_gt, test_mask)
                print('\tTest performance:', cur_test_perf)

                if val_loss / (self.test_index - self.valid_index) < best_valid_loss:
                    best_valid_loss = val_loss / (self.test_index - self.valid_index)
                    best_valid_perf = copy.deepcopy(cur_valid_perf)
                    best_valid_pred = val_pred.copy()
                    best_valid_gt = val_gt.copy()
                    best_valid_mask = val_mask.copy()
                    best_test_perf = copy.deepcopy(cur_test_perf)
                    best_test_pred = test_pred.copy()
                    best_test_gt = test_gt.copy()
                    best_test_mask = test_mask.copy()
                    print('Better valid loss:', best_valid_loss)
                    # self.save_model(model, f'../../data/pretrain/pretrain/{self.market_name}_ranklstm_model.pt')

            print('Epoch:', epoch, 'Time: %.4f' % (time() - t1))

        print('\nBest Valid performance:', best_valid_perf)
        print('\tBest Test performance:', best_test_perf)

        return best_valid_pred, best_valid_gt, best_valid_mask, best_valid_perf, \
               best_test_pred, best_test_gt, best_test_mask, best_test_perf

    def update_model(self, parameters):
        for name, value in parameters.items():
            self.parameters[name] = value
        return True

    def save_model(self, model, path):
        # Save both the model state and the parameters needed to reconstruct it
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': self.fea_dim,
            'hidden_units': self.parameters['unit']
        }, path)
        print(f"Model saved to {path}")

    def load_model(self, path):
        # Load the saved data
        checkpoint = torch.load(path, map_location=self.device)
        
        # Recreate the model architecture
        model = RankLSTMModel(
            input_dim=checkpoint['input_dim'],
            hidden_units=checkpoint['hidden_units']
        ).to(self.device)
        
        # Load the trained weights
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # Set to evaluation mode
        model.eval()
        print(f"Model loaded from {path}")
        return model

    def predict(self, model, start=None):
        model.eval()
        with torch.no_grad():
            test_pred = np.zeros([len(self.tickers), self.trade_dates - start], dtype=float)
            test_gt = np.zeros_like(test_pred)
            test_mask = np.zeros_like(test_pred)
    
            for offset in range(start - self.parameters['seq'] - self.steps + 1,
                                self.trade_dates - self.parameters['seq'] - self.steps + 1):
                eod_batch, mask_batch, price_batch, gt_batch = self.get_batch(offset)
                x = torch.tensor(eod_batch, dtype=torch.float32, device=self.device)
                mask = torch.tensor(mask_batch, dtype=torch.float32, device=self.device)
                base_price = torch.tensor(price_batch, dtype=torch.float32, device=self.device)
                gt = torch.tensor(gt_batch, dtype=torch.float32, device=self.device)
                pred = model(x)
                loss, reg_loss, rank_loss, return_ratio = self.compute_losses(pred, base_price, gt, mask,
                                                                             self.parameters['alpha'])
                idx = offset - (start - self.parameters['seq'] - self.steps + 1)
                test_pred[:, idx] = return_ratio.squeeze().cpu().numpy()
                test_gt[:, idx] = gt.squeeze().cpu().numpy()
                test_mask[:, idx] = mask.squeeze().cpu().numpy()

            performance = evaluate(test_pred, test_gt, test_mask)
                
            return (
                test_pred,
                test_gt,
                test_mask,
                performance
            )

## NASDAQ

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
data_path = '../../data/2013-01-01'
market_name = 'NASDAQ'
tickers_fname = f"{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv"
parameters = {'seq': 16, 'unit': 64, 'lr': 0.001, 'alpha': 0.1}

In [7]:
rank_lstm = RankLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    gpu=torch.cuda.is_available()
)

best_valid_pred, best_valid_gt, best_valid_mask, best_valid_perf, \
best_test_pred, best_test_gt, best_test_mask, best_test_perf = rank_lstm.train()

loaded_model = rank_lstm.load_model(f'../../data/pretrain/pretrain/{market_name}_ranklstm_model.pt')

#tickers selected: 1026
Raw prices shape: (1026, 1245)
single EOD data shape: (1245, 6)
device: cuda
Train Loss: 0.3880380899922268 0.38767044193841316 0.003676480199523384
Valid MSE: 0.34603505049433025 0.3455881202031696 0.0044693059005242375
	Valid performance: {'mse': np.float64(0.34841938198090655), 'mrrt': np.float64(0.02830520133997485), 'btl': np.float64(2.346037216950208), 'btl5': np.float64(1.909608857797866), 'btl10': np.float64(1.6359135491140473)}
Test MSE: 0.381060556771886 0.38068468824720586 0.0037586850493376137
	Test performance: {'mse': np.float64(0.38188139056291237), 'mrrt': np.float64(0.029925204020882726), 'btl': np.float64(2.4159301675263123), 'btl5': np.float64(1.7489471135077117), 'btl10': np.float64(1.4270304077886058)}
Better valid loss: 0.34603505049433025
Model saved to ../../data/pretrain/pretrain/NASDAQ_ranklstm_model.pt
Epoch: 0 Time: 9.3242
Train Loss: 0.3330870953766075 0.3326714783101469 0.004156173638546386
Valid MSE: 0.34316825795741307 0.342697740

In [8]:
print('Prediction performance:', best_test_perf)

Prediction performance: {'mse': np.float64(0.3712599877710443), 'mrrt': np.float64(0.021320617280677828), 'btl': np.float64(1.5484323463606415), 'btl5': np.float64(1.657204291175548), 'btl10': np.float64(1.5025656683711526)}


## NYSE

In [9]:
data_path = '../../data/2013-01-01'
market_name = 'NYSE'
tickers_fname = f"{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv"
parameters = {'seq': 16, 'unit': 32, 'lr': 0.001, 'alpha': 10}

In [11]:
rank_lstm = RankLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    gpu=torch.cuda.is_available()
)

best_valid_pred, best_valid_gt, best_valid_mask, best_valid_perf, \
best_test_pred, best_test_gt, best_test_mask, best_test_perf = rank_lstm.train()

loaded_model = rank_lstm.load_model(f'../../data/pretrain/pretrain/{market_name}_ranklstm_model.pt')

#tickers selected: 1737
Raw prices shape: (1737, 1245)
single EOD data shape: (1245, 6)
device: cuda
Train Loss: 0.45861445278734775 0.43340856932305 0.0025205883654999105
Valid MSE: 0.39834495988630114 0.3550583373696085 0.004328662208059714
	Valid performance: {'mse': np.float64(0.3560906645114033), 'mrrt': np.float64(0.005586772264850502), 'btl': np.float64(1.5103416894689872), 'btl5': np.float64(1.5025810444054504), 'btl10': np.float64(1.3603921068766789)}
Test MSE: 0.40138431364976906 0.36659014287880204 0.0034794171683623885
	Test performance: {'mse': np.float64(0.3678088094956644), 'mrrt': np.float64(0.008784298040469237), 'btl': np.float64(1.6099907967582112), 'btl5': np.float64(1.433299360025558), 'btl10': np.float64(1.3951632321077343)}
Better valid loss: 0.39834495988630114
Model saved to ../../data/pretrain/pretrain/NYSE_ranklstm_model.pt
Epoch: 0 Time: 7.7820
Train Loss: 0.36383151331463376 0.3303954494966043 0.003343606407508707
Valid MSE: 0.39855046700390556 0.3547630527

In [12]:
print('Prediction performance:', best_test_perf)

Prediction performance: {'mse': np.float64(0.36727901822826836), 'mrrt': np.float64(0.005959182182355945), 'btl': np.float64(1.4248405360267498), 'btl5': np.float64(1.1642507581260366), 'btl10': np.float64(1.120002801997363)}


## Embedding Generation

The `generate_and_save_embeddings` function takes the trained Rank-LSTM model and uses it as a feature extractor. For every stock and every trading day in the dataset, it feeds the historical sequence of features into the LSTM layer and captures the final hidden state. This hidden state is a dense vector, or "embedding," that encapsulates the learned temporal patterns of that stock up to that day.

These embeddings are then saved to a file. Instead of using a sequence of raw features as input, the next model (RRLSTM) will use these pre-computed embeddings, adding only the relational information to the prediction task.

In [13]:
def generate_and_save_embeddings(rank_lstm_instance, model_path, save_path):
    """
    Loads a trained RankLSTM model and uses it to generate and save the
    sequential embeddings for the entire dataset.
    """
    print(f"Loading trained RankLSTM model from: {model_path}")
    model = rank_lstm_instance.load_model(model_path)
    
    # The embedding is the output of model.lstm
    lstm_layer = model.lstm
    
    # Get parameters for iteration
    trade_dates = rank_lstm_instance.trade_dates
    seq_len = rank_lstm_instance.parameters['seq']
    num_stocks = len(rank_lstm_instance.tickers)
    hidden_units = rank_lstm_instance.parameters['unit']
    device = rank_lstm_instance.device

    # Prepare an array to store the embeddings for all stocks over all days
    all_embeddings = np.zeros((num_stocks, trade_dates, hidden_units), dtype=np.float32)

    print(f"Generating embeddings for {num_stocks} stocks over {trade_dates} days...")
    
    # Use torch.no_grad() for efficiency as we are not training
    with torch.no_grad():
        # Iterate through every single day in the dataset to generate the embedding for that day
        for day_index in range(trade_dates):
            if day_index < seq_len:
                # Not enough historical data to form a full sequence
                continue

            # Get the input feature sequence for the current day, from (day_index - seq_len) 
            eod_batch = rank_lstm_instance.eod_data[:, (day_index - seq_len) : day_index, :]
            
            # Convert to tensor
            x = torch.tensor(eod_batch, dtype=torch.float32, device=device)
            
            # Pass the data through the LSTM layer ONLY
            # lstm_out shape: (num_stocks, seq_len, hidden_units)
            lstm_out, _ = lstm_layer(x)
            
            # The embedding for the current day is the last output of the sequence
            # last_output shape: (num_stocks, hidden_units)
            last_output = lstm_out[:, -1, :]
            
            # Store it in our numpy array
            all_embeddings[:, day_index, :] = last_output.cpu().numpy()

            if (day_index + 1) % 100 == 0:
                print(f"  Processed {day_index + 1}/{trade_dates} days...")

    print(f"Finished generating embeddings. Final shape: {all_embeddings.shape}")
    
    # Save the final numpy array to disk
    np.save(save_path, all_embeddings)
    print(f"Embeddings successfully saved to: {save_path}")
    return all_embeddings

In [14]:
seq = parameters['seq']
unit = parameters['unit']

trained_model_path = f'../../data/pretrain/pretrain/{market_name}_ranklstm_model.pt'
embedding_save_path = f'../../data/pretrain/pretrain/{market_name}_rank_lstm_seq-{seq}_unit-{unit}_copy.csv.npy'

generate_and_save_embeddings(
    rank_lstm_instance=rank_lstm, 
    model_path=trained_model_path, 
    save_path=embedding_save_path
)

Loading trained RankLSTM model from: ../../data/pretrain/pretrain/NYSE_ranklstm_model.pt
Model loaded from ../../data/pretrain/pretrain/NYSE_ranklstm_model.pt
Generating embeddings for 1737 stocks over 1245 days...
  Processed 100/1245 days...
  Processed 200/1245 days...
  Processed 300/1245 days...
  Processed 400/1245 days...
  Processed 500/1245 days...
  Processed 600/1245 days...
  Processed 700/1245 days...
  Processed 800/1245 days...
  Processed 900/1245 days...
  Processed 1000/1245 days...
  Processed 1100/1245 days...
  Processed 1200/1245 days...
Finished generating embeddings. Final shape: (1737, 1245, 32)
Embeddings successfully saved to: ../../data/pretrain/pretrain/NYSE_rank_lstm_seq-16_unit-32_copy.csv.npy


array([[[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.99605745,  0.998528  ,  0.9943771 , ..., -0.9927821 ,
         -0.99383307, -0.9996425 ],
        [ 0.9962971 ,  0.99865854,  0.99474984, ..., -0.99351263,
         -0.9939586 , -0.9997383 ],
        [ 0.99667877,  0.99892396,  0.99544567, ..., -0.9954602 ,
         -0.995509  , -0.99983704]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.9934683 ,  0.9979189 ,  0.9851089 , ..., -